# Control Linkage — Run Notebook**This is the control panel.** You only edit four things:1. **CONFIG** — data paths, output paths, model, run parameters (cell 2)2. **PROMPT** — the linkage instructions (cell 4)3. **SCHEMA** — the response format (cell 5)4. Then **run the pipeline** (cells 6-9) and **compare runs** (cell 10)All the machinery lives in `control_linkage_engine.py` — you should not need to touch it.**Workflow for prompt development:**- change the PROMPT and/or `run_name`, run cells 6-9, look at the scorecard.- each run saves verdicts + a manifest under `output_dir` tagged by `run_name`.- use cell 10 to compare scorecards across runs.

## 1. Setup — imports, gateway client, engine

In [ ]:
import asyncioimport pandas as pdfrom control_linkage_engine import LinkageConfig, LinkageEngine# --- YOUR GATEWAY CLIENT + TOKEN CACHE ---# These come from your environment setup. `oai` = async OpenAI-compatible client,# `tok` = your TrustTokenCache with async .get() and .invalidate().# (Paste your existing client-init code here, or import it.)## from my_gateway_setup import oai, tokassert 'oai' in dir() and 'tok' in dir(), "Define `oai` (client) and `tok` (token cache) first."print("gateway client ready")

## 2. CONFIG — set your paths and parameters hereThis is the main thing you edit per environment. Everything is in one place.

In [ ]:
cfg = LinkageConfig(    # ---- INPUT DATA PATHS ----    rapid2_path      = "../data/Rapid2_REGDEV_US_2020_2025.parquet",    regmap_ctrl_path = "../data/regmap_ctrl.parquet",          # RegMap RSM->control    helios_path      = "../data/Helios/helios_controls.parquet",    golden_path      = "../data/golden_control_links.parquet",    rwt_path         = "../data/results_with_truth.parquet",   # prior RSM run (linked_rsm_ids)    # ---- OUTPUT ----    output_dir       = "../output/control_linkage",    run_name         = "v2_6_baseline",     # <-- CHANGE per experiment (tags all outputs)    # ---- MODEL ----    model            = "gpt-5.6-luna",    use_case         = "UC0003056",    is_gemini        = False,               # True for gemini models (changes call kwargs)    # ---- JUDGING PARAMS ----    batch_size       = 15,                  # controls per LLM call (lower = fewer parse errors)    concurrency      = 20,                  # simultaneous calls (raise to ~40 to go faster)    save_every       = 100,                 # checkpoint every N batches (restart safety)    # ---- SCOPE ----    scope_to_golden_records = True,         # judge only records that are in the golden    # ---- COLUMN NAMES (override only if your data differs) ----    col_record_id="RECORD_ID", col_reg_id="REG_ID", col_sum_id="SUM_ID",    col_l1_ctrl="L1_CTRL", col_final_decision="final_decision",    col_linked_rsm_ids="linked_rsm_ids",)print("run:", cfg.run_name, "| output ->", cfg.output_dir)

## 3. Build the engine and load data

In [ ]:
eng = LinkageEngine(cfg, oai, tok)data = eng.load_data()# Optional: custom text builders if the defaults don't fit your columns:#   data = eng.load_data(alert_text_builder=my_fn, control_text_builder=my_fn)

## 3b. DATA STORY  (run before judging — this is the funnel/coverage narrative)Explains the data itself: sizes, overlaps, reachability ceiling, linkage coverage,and the two data-quality issues (missing RSM, RGL-with-no-controls). Present this tobusiness to frame *why* the numbers read the way they do.

In [ ]:
from control_linkage_data_story import (data_story, source_sizes, overlaps,    reachability, linkage_coverage, missing_rsm_impact, rgl_no_controls_impact)# full narrative in one call:story = data_story(eng)# or run any section individually:# source_sizes(eng); overlaps(eng); reachability(eng)# linkage_coverage(eng); missing_rsm_impact(eng); rgl_no_controls_impact(eng)

## 4. PROMPT  ← the team develops thisEdit the linkage logic here. This is the system prompt sent for every batch.

In [ ]:
PROMPT = """You are a Lead Operational Risk & Controls AI Architect at HSBC. You are given ONEregulatory ALERT and a LIST of internal CONTROLS. For EACH control, decide whether the alertgenuinely impacts what the control must DO.READ THE WHOLE CONTROL - it has two layers, use BOTH:1. THE SPECIFIC TASK - Title + What/How/Why (the concrete activity).2. THE LIBRARY OBJECTIVE - the broader perimeter it rolls up to.An alert can link via EITHER layer.LINK when the alert would change this control's procedure, scope, threshold/timing, or when itssubject falls squarely within the library objective. A shared GENERIC objective with no concreteoverlap is NOT a link. Do not link against your own reasoning: if nothing concrete overlaps,NOT_LINKED.For EACH control return: decision, relevance_score (0-10), confidence_score (1-5), short evidencesubstrings, and 1-2 sentence reasoning written BEFORE the decision.Return ONLY JSON matching the provided schema, one item per control, id copied verbatim."""print("prompt length:", len(PROMPT.split()), "words")

## 5. RESPONSE SCHEMA  ← the team defines the output shape`strict: True` guarantees every key is present. Field descriptions double as instructions.

In [ ]:
SCHEMA = {    "type": "json_schema",    "json_schema": {        "name": "control_linkage", "strict": True,        "schema": {            "type": "object", "additionalProperties": False, "required": ["items"],            "properties": {"items": {                "type": "array",                "description": "One entry per control supplied, same order, no omissions/duplicates.",                "items": {                    "type": "object", "additionalProperties": False,                    "required": ["l1_control_id","decision","relevance_score","confidence_score",                                 "evidence_from_alert","evidence_from_control","reasoning"],                    "properties": {                        "l1_control_id":  {"type":"string","description":"Control id, copied VERBATIM from input."},                        "decision":       {"type":"string","enum":["LINKED","NOT_LINKED","INSUFFICIENT_EVIDENCE"],                                           "description":"LINKED=alert affects what the control must do. NOT_LINKED=different domain/obligation. INSUFFICIENT_EVIDENCE=text too sparse to decide (not a hedge for weak links)."},                        "relevance_score":{"type":"integer","description":"Link strength 0-10. 8-10 direct/material; 5-7 real but partial/indirect; 1-4 tangential(=NOT_LINKED); 0 unrelated. Make it discriminating."},                        "confidence_score":{"type":"integer","description":"Certainty 1-5 by how explicitly the texts support the verdict. Independent of relevance_score."},                        "evidence_from_alert":  {"type":["string","null"],"description":"Verbatim substring from the ALERT, or null."},                        "evidence_from_control":{"type":["string","null"],"description":"Verbatim substring from the CONTROL (cite Library Objective if the link rests there), or null."},                        "reasoning":      {"type":["string","null"],"description":"1-2 sentences: the concrete basis for the decision, written before deciding."}                    }                }            }}        }    }}print("schema ready")

## 6. Build candidates (controls under each alert's linked RSMs)

> **Upstream provision:** if your alert->RGL->RSM mapping (RWT) changes format,> write a function `my_adapter(rwt_df) -> {record_id: set(rsm_ids)}` and call> `eng.build_candidates(rsm_adapter=my_adapter)`. Nothing else changes.

In [ ]:
cands = eng.build_candidates()

## 7. Judge  (batched + parallel + checkpointed)Safe to re-run after a restart — it resumes from the checkpoint automatically.This is the long step. Watch the progress bar.

In [ ]:
verdicts = await eng.judge(PROMPT, SCHEMA, resume=True)

## 8. Score vs golden

In [ ]:
report = eng.score(verdicts)eng.cost()   # adjust prices inside if needed

## 9. Save (verdicts + manifest with metrics, config, prompt hash)

In [ ]:
eng.save(verdicts, report, prompt=PROMPT, schema=SCHEMA)

## 10. Compare runsLoad the manifests of several runs and compare their scorecards side by side.This is how you decide which prompt wins.

In [ ]:
import json, glob, pandas as pdrows = []for mpath in glob.glob(cfg.output_dir.rstrip("/") + "/*_manifest.json"):    with open(mpath) as f:        m = json.load(f)    met = m.get("metrics", {})    rows.append({"run": m["run_name"], "timestamp": m["timestamp"],                 "recall": met.get("recall"), "validated": met.get("validated"),                 "addressable": met.get("addressable"), "discoveries": met.get("discoveries"),                 "prompt_sha1": (m.get("prompt_sha1") or "")[:8]})comparison = pd.DataFrame(rows).sort_values("timestamp")comparison["recall"] = (comparison["recall"]*100).round(1).astype(str) + "%"display(comparison)

---## 11. ANALYSIS  (run after a finished run)All analysis lives in `control_linkage_analysis.py`. Each function takes `(eng, verdicts)`.Run them individually, or `full_analysis(...)` to do everything at once.

In [ ]:
from control_linkage_analysis import diagnose, recall_at_k, audit, review, waterfall, full_analysis

### 11a. Diagnose — where recall is lost (candidate-gen vs judge)

In [ ]:
diag = diagnose(eng, verdicts)

### 11b. Recall@K — SME shortlist coverage

In [ ]:
rk = recall_at_k(eng, verdicts)

### 11c. LLM-as-judge audit`false_negatives` = are the model's rejections correct? `false_positives` = are discoveries genuine?(Uses the engine's gateway; clears no state.)

In [ ]:
audit_fn = await audit(eng, verdicts, which="false_negatives", sample=1000)audit_fp = await audit(eng, verdicts, which="false_positives", sample=1000)

### 11d. Manual review — read full cases (optionally filtered to an audit bucket)

In [ ]:
# false negatives the audit called a true miss:_ = review(eng, verdicts, which="FN", n=15, audit_df=audit_fn, only_bucket="TRUE_LINK_MISSED")# discoveries to eyeball:# _ = review(eng, verdicts, which="FP", n=15)

### 11e. Waterfall (deck-ready PNG)

In [ ]:
fig = waterfall(eng, verdicts, report, audit_fn_df=audit_fn, save_path=cfg.resolved("waterfall.png"))

### 11f. One-shot — run all analysis at onceDiagnose + recall@K + FN/FP audit + waterfall, returned in one dict.

In [ ]:
results = await full_analysis(eng, verdicts, report, run_audit=True, audit_sample=1000, save_dir=cfg.output_dir)

---## 12. L1C (Library Control) AGGREGATION  — independent, additional to 1CRolls the 1C verdicts up to library-control (L1C) level: **an L1C is LINKED if ANY of itschild 1C controls is linked** (max rule). Scores independently against the golden's L1C truth.**The 1C results above are untouched** — this is an additional, separate view.

In [ ]:
from control_linkage_analysis import aggregate_to_l1c, score_l1c# 1. roll 1C verdicts up to L1C (any child linked -> L1C linked; score = max child)l1c_verdicts = aggregate_to_l1c(eng, verdicts, l1c_col="L1_LIB_CTRL")# 2. independent L1C scorecard vs golden L1C truthreport_l1c = score_l1c(eng, l1c_verdicts, l1c_col="L1_LIB_CTRL")# 3. save the L1C results separatelyl1c_verdicts.to_parquet(cfg.resolved("verdicts_L1C.parquet"), index=False)

> If the L1C column names differ, pass them explicitly:> `aggregate_to_l1c(eng, verdicts, l1c_col="<helios L1C col>")` and> `score_l1c(eng, l1c_verdicts, l1c_col="<helios L1C col>", golden_l1c_col="<golden L1C col>")`.> The functions auto-detect library-control columns and will list candidates if not found.